# Figure 4e,f. Fine-mapping GTEx with ADSP and UK Biobank LD

Full GTEx summary statistics from `07` `[multi_ref_simulation]` (`2026_output/Gtex_simulation`, n = 838), fine-mapped with the ADSP sketch, the UKB sketch, and their mixture. Pass both sketches to `susie_rss(X = list(X_ADSP, X_UKB))` to estimate the weights ω and fit the weighted reference in one call. The earlier MSK notebook `2026_Simulation/5.A.Multi_Gtex_Simulation` estimated the weights and built the reference in separate steps.

- `[mix_finemap]`: SuSiE-RSS (`ADSP`, `UKB`, `mix`) and SuSiE-rRSS (`*_rrss`: `R_finite = 10000` per panel as in `09`, `R_mismatch = "eb_mix"`; the mixture uses B_eff = 1 / Σ ω²/B). First align both sketches to the GTEx allele coding. h2g 0.05 / 0.1 / 0.15 (nh² = 40 / 80 / 120), 1–3 causal.
- `[mix_summary]`: power / FDR counts per fit and SER fallback model, fallback diagnostics, ADSP weight ω₁ of `mix` and `mix_rrss`.

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_ADSP_UKB_mix/` (`result/`, `summary/result_mix_summary.rds`), read by `Data_Processing/Figure_4/Fig4E_4F_S9C.GTEx_Mix`.

## Fine-mapping

In [ ]:
[mix_finemap]
# Full GTEx summary statistics (07 [multi_ref_simulation]) fine-mapped with the ADSP and UKB sketches,
# each alone and as a mixture (list of sketches -> susie_rss estimates omega and refits on the weighted
# reference in one call). SuSiE-RSS and SuSiE-rRSS (R_finite, R_mismatch = "eb_mix") for each LD input.
parameter: genofile = paths
# 07 output: Gtex_simulation/result/h2g_<h2g>/causal_<k>/sample_*_multi_ref.rds
parameter: cwd = path("output")
parameter: R_finite = 10000
# sketch B for SuSiE-rRSS, as B_inf in 09 [finemap]; one value per panel for the mixture
parameter: job_size = 30
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
input: genofile, group_by = 1
output: f'{cwd:a}/{_input[0]:bn}.rds'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(plink2R)
    library(tidyverse)
    library(MASS)
    library(crayon)
    library(pecotmr)
    library(susieR)
    library(vroom)
    library(pgenlibr)
    library(Rfast)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)

    # pecotmr GitHub version (conda not yet updated)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    # Project utilities
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/finemap_qc_ver2.r")
    source_build <- function(dir) {
      env <- new.env(parent = globalenv())
      for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
      env
    }
    susie_hc <- source_build("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/")$susie_rss

    # Safe wrapper: takes the susie fn explicitly, catches errors, returns NULL
    safe_susie <- function(.fn, .label, ...) {
      tryCatch(
        withCallingHandlers(
          .fn(...),
          warning = function(w) invokeRestart("muffleWarning")
        ),
        error = function(e) {
          cat(sprintf("[FAIL] %s: %s\n", .label, conditionMessage(e)))
          NULL
        }
      )
    }

    # -----------------------------------------------------------------------------
    # Load inputs
    # -----------------------------------------------------------------------------
    path <- ${_input:r}
    dat  <- readRDS(path)
    n    <- dat$n_samples
    B    <- ${R_finite}

    # -----------------------------------------------------------------------------
    # Put both sketches on the GTEx (target) allele coding.
    # 07 re-QCs each panel against the simulated z (flip_cols = TRUE): its z is in the
    # panel's coding and its U rows follow its sumstat rows. A row is flipped back when
    # the panel z has the opposite sign of the GTEx z, so z and both sketches share one coding.
    # -----------------------------------------------------------------------------
    ins  <- dat$insample_input$sumstat              # GTEx coding, Xmat column order (= true_vars indexing)
    keep <- ins$variants_id_original[ins$variants_id_original %in%
              intersect(dat$ADSP_input$sumstat$variants_id_original, dat$UKB_input$sumstat$variants_id_original)]
    z    <- ins$z[match(keep, ins$variants_id_original)]

    to_target <- function(inp) {
      idx <- match(keep, inp$sumstat$variants_id_original)
      zp  <- inp$sumstat$z[idx]
      s   <- ifelse(abs(zp + z) < abs(zp - z), -1, 1)      # -1 where the panel flipped the allele
      U   <- inp$U[idx, , drop = FALSE] * s                 # p x B
      rownames(U) <- keep
      list(X = t(U), n_flip = sum(s < 0))                   # B x p
    }
    ADSP_ref <- to_target(dat$ADSP_input)
    UKB_ref  <- to_target(dat$UKB_input)
    X_ADSP   <- ADSP_ref$X
    X_UKB    <- UKB_ref$X

    true_vars <- match(ins$variants_id_original[dat$true_vars], keep)
    if (anyNA(true_vars)) cat(sprintf("[NOTE] %d causal variant(s) not in both panels\n", sum(is.na(true_vars))))
    true_vars <- true_vars[!is.na(true_vars)]
    cat(sprintf("p = %d of %d GTEx variants; flipped rows ADSP = %d, UKB = %d; B ADSP = %d, UKB = %d\n",
                length(keep), nrow(ins), ADSP_ref$n_flip, UKB_ref$n_flip, nrow(X_ADSP), nrow(X_UKB)))

    # -----------------------------------------------------------------------------
    # Fits: SuSiE-RSS and SuSiE-rRSS for ADSP, UKB and the ADSP + UKB mixture
    # -----------------------------------------------------------------------------
    ADSP      <- safe_susie(susie_hc, "ADSP",      z = z, X = X_ADSP, n = n)
    UKB       <- safe_susie(susie_hc, "UKB",       z = z, X = X_UKB,  n = n)
    mix       <- safe_susie(susie_hc, "mix",       z = z, X = list(X_ADSP, X_UKB), n = n)

    ADSP_rrss <- safe_susie(susie_hc, "ADSP_rrss", z = z, X = X_ADSP, n = n,
                            R_finite = B, R_mismatch = "eb_mix")
    UKB_rrss  <- safe_susie(susie_hc, "UKB_rrss",  z = z, X = X_UKB,  n = n,
                            R_finite = B, R_mismatch = "eb_mix")
    mix_rrss  <- safe_susie(susie_hc, "mix_rrss",  z = z, X = list(X_ADSP, X_UKB), n = n,
                            R_finite = B, R_mismatch = "eb_mix")

    # -----------------------------------------------------------------------------
    # Collect fits
    # -----------------------------------------------------------------------------
    result_list <- list(
      ADSP      = ADSP,
      UKB       = UKB,
      mix       = mix,
      ADSP_rrss = ADSP_rrss,
      UKB_rrss  = UKB_rrss,
      mix_rrss  = mix_rrss
    )

    output <- list(
      result_list   = result_list,
      omega_weights = list(mix = mix$omega_weights, mix_rrss = mix_rrss$omega_weights),   # [1] ADSP, [2] UKB
      n_sample      = n,
      n_snp         = length(keep),
      R_finite      = B,
      input_path    = path,
      true_vars     = true_vars
    )
    saveRDS(output, ${_output:r})

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
job_name="Gtex_ADSP_UKB_mix"
job_dir="${base_dir}/${job_name}"

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base BSUB script template (named as job_name) =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME_c_CAUSAL_h_H2G
#BSUB -n 5
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 72:00
#BSUB -o /WORK_DIR/log/JOB_NAME_c_CAUSAL_h_H2G.%J.out
#BSUB -e /WORK_DIR/log/JOB_NAME_c_CAUSAL_h_H2G.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/27.Fig4E_4F.GTEx_ADSP_UKB_Mix.ipynb" mix_finemap \
    --genofile `ls /sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_simulation/result/h2g_H2G/causal_CAUSAL/*.rds` \
    --cwd WORK_DIR/result/h2g_H2G/causal_CAUSAL -j 5
EOF

chmod +x ${job_name}.sh

base_sh=${job_name}.sh
work_dir="${job_dir}"

# nh2 = 800 x h2g = 40 / 80 / 120 (Fig 4e)
for causal in 1 2 3; do
  for h2g in 0.05 0.1 0.15; do
      output_script="${job_name}_causal_${causal}_h2g_${h2g}.sh"
      sed -e "s|CAUSAL|${causal}|g" \
          -e "s|WORK_DIR|${work_dir}|g" \
          -e "s|H2G|${h2g}|g" \
          -e "s|JOB_NAME|${job_name}|g" \
        "${base_sh}" > "${output_script}"

        sb "${output_script}"
  done
done

## Summary

In [ ]:
[mix_summary]
# One row per (simulation, fit) and per SER fallback sub-model ("<fit>_SER"), with the
# fallback diagnostics and the estimated ADSP weight of both mixture fits.
parameter: folder = path
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "50h"
parameter: mem = "30G"
parameter: numThreads = 1
parameter: container = ""
output: f'{cwd:a}/{folder:b}_{step_name}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads, tags = f'{step_name}_{_output[0]:bn}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    `%||%` <- function(a, b) if (is.null(a)) b else a

    # true_set_num / discover_var_num as in 10 [susie_bias_summary]
    true_set_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      sum(vapply(cs, function(x) length(intersect(variant, x)) > 0, logical(1)))
    }
    discover_var_num <- function(variant, cs) {
      if (length(cs) == 0) return(0L)
      length(intersect(variant, unlist(cs)))
    }

    path  <- ${folder:r}
    files <- list.files(path, pattern = "sample_.*\\.rds$", recursive = TRUE, full.names = TRUE)

    tb <- map_dfr(files, function(f) {
      res     <- readRDS(f)
      variant <- res$true_vars
      ncausal <- as.numeric(sub(".*ncausal_([0-9.]+).*", "\\1", f))
      w_rss   <- res$omega_weights$mix[1]      %||% NA_real_
      w_rrss  <- res$omega_weights$mix_rrss[1] %||% NA_real_

      row_from_fit <- function(fit, label, diag) {
        cs <- fit$sets$cs
        tibble(
          input_file         = f,
          method             = label,
          ncausal            = ncausal,
          true_var_number    = discover_var_num(variant, cs),
          true_set_number    = true_set_num(variant, cs),
          total_var_number   = length(variant),
          total_set_number   = length(cs),
          total_set_length   = length(unlist(cs)),
          Q_art              = diag$Q_art              %||% NA_real_,
          artifact_evaluable = diag$artifact_evaluable %||% NA,
          D_CS_max           = if (!is.null(diag$bf_attenuation$cs_max))
                                 suppressWarnings(max(diag$bf_attenuation$cs_max, na.rm = TRUE)) else NA_real_,
          R_reliability_flag = diag$R_reliability_flag %||% NA,
          ADSP_weight        = w_rss,
          ADSP_weight_rrss   = w_rrss
        )
      }

      map_dfr(names(res$result_list), function(m) {
        sus <- res$result_list[[m]]
        if (is.null(sus)) return(NULL)                       # safe_susie NULLs
        diag <- sus$R_finite_diagnostics
        rows <- row_from_fit(sus, m, diag)
        if (!is.null(diag$ser_model))
          rows <- bind_rows(rows, row_from_fit(diag$ser_model, paste0(m, "_SER"), diag$ser_model$R_finite_diagnostics))
        rows
      })
    })
    saveRDS(tb, ${_output:r})

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Gtex_ADSP_UKB_mix/"
mkdir -p ${work_dir}/summary
cd ${work_dir}/summary

# Create the base_script file and write the bash code into it
cat << 'EOF' > base_script
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J mix_summary
#BSUB -n 2
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=30000]
#BSUB -W 16:00
#BSUB -o WORK_DIR/summary/summary.%J.out
#BSUB -e WORK_DIR/summary/summary.%J.err


export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/27.Fig4E_4F.GTEx_ADSP_UKB_Mix.ipynb" mix_summary \
    --folder WORK_DIR/result/ \
    --cwd WORK_DIR/summary


EOF

base_sh="base_script"

output_script="summary_mix.sh"
cat ${base_sh} |sed "s|WORK_DIR|${work_dir}|g"    > ${output_script}
sb ${output_script}